<a href="https://colab.research.google.com/github/craljimenez/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/blob/main/clases/semana04-clase09-agrupaciones-groupby/notebooks/lab-agrupaciones-groupby.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: Agrupaciones y análisis con groupby
### Semana 5 · Clase 9
**Curso:** Análisis y Visualización de Datos para la Toma de Decisiones (Esumer)

En la Clase 8 dejamos listo `membresias_gimnasio_limpio.csv`: una tabla sin duplicados, con tipos correctos y con columnas nuevas de segmentación (`ciudad`, `plan`, `grupo_edad`, `activo`, `trimestre_inscripcion`...).

Hoy usamos esa tabla para responder la pregunta que casi siempre aparece en un análisis de negocio: **¿cómo se comporta cada segmento?** ¿Cuántos socios hay en cada ciudad? ¿Qué plan prefiere cada grupo de edad? ¿Cómo cambian las inscripciones mes a mes? La herramienta para eso es `groupby`.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

socios = pd.read_csv("membresias_gimnasio_limpio.csv")
socios["fecha_inscripcion"] = pd.to_datetime(socios["fecha_inscripcion"])
print(socios.shape)
socios.dtypes

Antes de empezar, fíjate en dos cosas del resultado de `dtypes`: `fecha_inscripcion` es una fecha de verdad (la convertimos porque al leer el CSV llega como texto), y `activo` es `bool`, así que sirve directamente para filtrar y agrupar.

## 1. La idea: dividir, aplicar, combinar

Casi todas las preguntas por segmento siguen el mismo patrón, que se conoce como **split-apply-combine**:

| Paso | Qué hace | En pandas |
|---|---|---|
| **Dividir** (split) | separa las filas en grupos según una columna | `groupby("ciudad")` |
| **Aplicar** (apply) | calcula algo dentro de cada grupo | `.size()`, `.mean()`, `.agg(...)` |
| **Combinar** (combine) | junta los resultados en una sola tabla | pandas lo hace por ti |

En la Clase 4 hicimos esto a mano: sacamos los valores únicos con `.unique()`, filtramos uno por uno y sumábamos dentro de un `for`. `groupby` hace exactamente lo mismo, pero en una sola línea y sin errores de copia.

In [ ]:
grupos = socios.groupby("ciudad")
print(type(grupos))
grupos.size()

Fíjate en el primer `print`: `groupby` **no devuelve una tabla** todavía, devuelve un objeto de grupos (`DataFrameGroupBy`). Es una especie de receta a la espera de que le digas qué calcular dentro de cada grupo. Por eso `.size()` (contar filas por grupo) es el paso que convierte la receta en resultado.

Para ver qué hay dentro de cada grupo, puedes recorrerlo como lo harías con un `for`:

In [ ]:
for ciudad, subtabla in socios.groupby("ciudad"):
    print(f"{ciudad}: {len(subtabla)} socios")

Cada `subtabla` es un `DataFrame` completo con solo las filas de esa ciudad. Recorrer el grupo es útil para entenderlo, pero para resumir casi siempre es más corto aplicar directamente una agregación, como veremos ahora.

## 2. Agregaciones simples por grupo

Una agregación resume cada grupo en un solo número. Las más usadas son las que ya conoces de la Clase 2 y de la Clase 4, pero ahora aplicadas por grupo:

In [ ]:
socios.groupby("ciudad")["edad"].mean()

In [ ]:
print(socios.groupby("ciudad")["edad"].sum())
print()
print(socios.groupby("ciudad")["edad"].max())

Ojo con un detalle que confunde a mucha gente: `.size()` y `.count()` **no son lo mismo**.

- `.size()` cuenta **filas** en cada grupo, tengan o no datos vacíos.
- `.count()` cuenta **valores no nulos** de una columna específica.

Compáralos con `telefono`, que tiene algunos vacíos:

In [ ]:
pd.DataFrame({
    "filas (size)": socios.groupby("ciudad").size(),
    "teléfonos no vacíos (count)": socios.groupby("ciudad")["telefono"].count(),
})

En Medellín hay 22 socios, pero solo 18 tienen teléfono registrado. Si calculas un promedio o un conteo sobre `telefono`, pandas ignora los vacíos automáticamente, así que el denominador cambia. Por eso conviene siempre preguntarte **sobre cuántos registros** se calculó cada número.

## 3. Varias agregaciones a la vez

Casi nunca necesitas un solo número por grupo. Para armar un resumen con varias columnas de una vez, `.agg()` acepta una **agregación con nombre**: escribes el nombre que quieres en la tabla final y, entre paréntesis, la columna y la función a aplicar.

In [ ]:
resumen_ciudad = socios.groupby("ciudad").agg(
    socios=("id_socio", "count"),
    edad_promedio=("edad", "mean"),
    edad_maxima=("edad", "max"),
    dias_antiguedad_promedio=("antiguedad_dias", "mean"),
)
resumen_ciudad.round(1)

Cada fila del resumen es una ciudad y cada columna es un indicador. El nombre de la izquierda (`socios`, `edad_promedio`...) es solo una etiqueta que tú eliges; lo que importa es la pareja `("columna", "función")` de la derecha.

También puedes aplicar varias funciones a la misma columna con una lista:

In [ ]:
socios.groupby("ciudad")["edad"].agg(["count", "mean", "median", "min", "max"]).round(1)

Observa la columna `median`: en **todas** las ciudades aparece 53. Esto no es una casualidad del análisis, es un efecto de la limpieza de la Clase 8: los 8 valores de `edad` que estaban mal escritos o vacíos se imputaron con la mediana (53), y esa mediana se repite muchas veces dentro de cada grupo. Un resultado que se repite así no es un hallazgo; es una señal de que el dato fue rellenado. Vuelve a esta idea en la Sección 8.

### Funciones de agregación disponibles

Las funciones que puedes escribir como texto dentro de `.agg()` (o pasar directamente a `.groupby().agg(...)`) son las siguientes:

| Función | Qué calcula |
|---|---|
| `"count"` | número de valores no nulos |
| `"size"` | número de filas del grupo, tengan o no datos vacíos |
| `"sum"` | suma |
| `"mean"` | promedio |
| `"median"` | mediana |
| `"min"` / `"max"` | mínimo / máximo |
| `"std"` / `"var"` | desviación estándar / varianza |
| `"sem"` | error estándar de la media |
| `"nunique"` | número de valores distintos |
| `"first"` / `"last"` | primer / último valor no nulo del grupo |
| `"prod"` | producto de los valores |

Para cálculos que no están en la lista, puedes escribir tu propia función con `lambda`. Por ejemplo, el percentil 90 de la edad por ciudad:

```python
socios.groupby("ciudad").agg(edad_p90=("edad", lambda s: s.quantile(0.9)))
```

Documentación oficial para seguir explorando:

- [Guía de agregación de pandas (user guide)](https://pandas.pydata.org/docs/user_guide/groupby.html#aggregation)
- [Referencia completa de funciones de groupby](https://pandas.pydata.org/docs/reference/groupby.html)
- [Referencia de `DataFrameGroupBy.agg`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)

## 4. Agrupar por dos o más columnas

Para cruzar dos segmentos, pasa una lista de columnas. Por ejemplo, cuántos socios hay en cada combinación de ciudad y plan:

In [ ]:
conteo_ciudad_plan = socios.groupby(["ciudad", "plan"]).size()
conteo_ciudad_plan

El resultado tiene **dos niveles de índice** (ciudad y plan), lo que pandas llama un `MultiIndex`. Es legible, pero para comparar es mucho más cómodo tener una tabla de doble entrada: `.unstack()` convierte el segundo nivel del índice en columnas.

In [ ]:
tabla_cruzada = socios.groupby(["ciudad", "plan"]).size().unstack(fill_value=0)
tabla_cruzada

`fill_value=0` pone un cero donde una combinación no existe (en Itagüí, por ejemplo, nadie tiene plan Trimestral), en vez de dejar un vacío.

Si prefieres una tabla normal, sin índice especial, usa `.reset_index()`. Ya viste este método en la Clase 7 (sección de índice): devuelve el índice a columnas normales.

In [ ]:
conteo_plano = socios.groupby(["ciudad", "plan"]).size().reset_index(name="socios")
conteo_plano

Existe también un atajo para tablas cruzadas: `pd.crosstab()`. Hace lo mismo que `groupby` + `unstack` y, si le pides `margins=True`, agrega los totales:

In [ ]:
pd.crosstab(socios["ciudad"], socios["plan"], margins=True)

## 5. Ordenar y construir una tabla resumen

Un resumen es útil cuando está ordenado de forma que responda una pregunta concreta. Por ejemplo, ¿qué ciudad tiene más socios? Ordena por la columna que te interesa con `.sort_values()`:

In [ ]:
resumen_ciudad.sort_values("socios", ascending=False).round(1)

Piensa en la tabla resumen como un **cuadro de indicadores**: pocas filas (los segmentos), pocas columnas (los indicadores), y un orden que responde a la pregunta de negocio. Un cuadro bien armado se lee de arriba hacia abajo sin explicación adicional.

## 6. Porcentajes y participaciones

Un conteo dice cuántos socios hay; una participación dice **qué parte** representan. Hay dos preguntas distintas y conviene no confundirlas:

1. ¿Qué parte del total representa cada plan? (participación sobre el total)
2. Dentro de cada ciudad, ¿qué parte de los socios tiene cada plan? (participación dentro del grupo)

Para la primera, `value_counts(normalize=True)` hace el trabajo:

In [ ]:
(socios["plan"].value_counts(normalize=True) * 100).round(1)

Para la segunda, necesitamos dividir cada conteo entre el total **de su ciudad**. Aquí aparece un método nuevo: `.transform()`. A diferencia de `.sum()`, que devuelve un número por grupo, `.transform("sum")` devuelve un valor por cada fila del grupo (el total de su ciudad), así que puedes dividir directamente sin perder el tamaño de la tabla.

In [ ]:
conteo = socios.groupby(["ciudad", "plan"]).size()
total_por_ciudad = conteo.groupby(level="ciudad").transform("sum")
participacion = (conteo / total_por_ciudad * 100).round(1)
participacion

Lee el resultado como: *"en Envigado, el 61,5 % de los socios tiene plan Mensual"*. Observa que cada bloque de ciudad suma 100.

`transform` también sirve para agregar una columna con un valor de grupo a cada fila de la tabla original, sin reducir sus filas. Por ejemplo, el tamaño de la ciudad de cada socio:

In [ ]:
socios["socios_en_su_ciudad"] = socios.groupby("ciudad")["id_socio"].transform("count")
socios[["nombre", "ciudad", "socios_en_su_ciudad"]].head(5)

## 7. Comparar categorías y periodos

### 7.1 Comparar categorías

Una pregunta típica es si dos grupos se diferencian en algo. Por ejemplo, ¿son distintas las edades de los socios activos y los inactivos?

In [ ]:
socios.groupby("activo")["edad"].agg(["count", "mean"]).round(1)

Con `pd.crosstab()` también puedes cruzar dos categorías: aquí, cuántos socios de cada grupo de edad están activos.

In [ ]:
pd.crosstab(socios["grupo_edad"], socios["activo"], margins=True)

Nota el tamaño de las celdas: hay 8 socios inactivos en total. Con grupos tan pequeños, cualquier diferencia de promedios puede ser ruido, no un patrón real. Esta idea la trabajamos en la Sección 8.

### 7.2 Comparar periodos

Para comparar periodos de tiempo, agrupa por una fecha convertida a periodo. `dt.to_period("M")` convierte cada fecha a su mes (2026-03, 2026-04...), y `dt.quarter` (que ya tenemos como `trimestre_inscripcion`) lo convierte a trimestre.

In [ ]:
inscripciones_mes = socios.groupby(socios["fecha_inscripcion"].dt.to_period("M")).size()
inscripciones_mes

In [ ]:
socios.groupby("trimestre_inscripcion").agg(
    inscripciones=("id_socio", "count"),
    edad_promedio=("edad", "mean"),
).round(1)

Compara los dos resultados: uno tiene 13 meses y el otro solo 4 trimestres, pero ambos cuentan exactamente los mismos 50 socios. Con más detalle se ven más variaciones; con menos, se ve la tendencia general. Elegir la granularidad correcta depende de la pregunta.

## 8. Cuidado al sumar cosas medidas en periodos distintos

En la Clase 8 creamos `precio_plan_cop`, que es lo que paga el socio por su plan completo: 90.000 por un mes, 240.000 por tres meses y 800.000 por un año. Si sumas esa columna por ciudad, estarías mezclando un mes con un año.

La forma correcta es llevar todo a la **misma escala** antes de agregar. Aquí, a un valor mensual equivalente:

In [ ]:
meses_del_plan = {"Mensual": 1, "Trimestral": 3, "Anual": 12}
socios["meses_del_plan"] = socios["plan"].map(meses_del_plan)
socios["precio_mensual_equivalente"] = socios["precio_plan_cop"] / socios["meses_del_plan"]

socios[["plan", "precio_plan_cop", "meses_del_plan", "precio_mensual_equivalente"]].drop_duplicates("plan")

In [ ]:
socios.groupby("ciudad").agg(
    socios=("id_socio", "count"),
    ingreso_mensual_equivalente=("precio_mensual_equivalente", "sum"),
).round(0)

Ojo con la interpretación: esto es un **ingreso mensual equivalente estimado**, no ingreso real. No tenemos fechas de pago ni sabemos si el socio renovó, así que el número describe el valor de la cartera de planes, no lo que entró a caja. Decirlo con esa precisión es parte del análisis.

## 9. Leer los resultados sin sobreinterpretarlos

Un promedio sin su tamaño de muestra es incompleto. Mira el conteo de Itagüí en la tabla por ciudad: solo tiene 2 socios. Su promedio existe, pero con dos personas cualquier cosa puede pasar. Una regla práctica: **no compares grupos con muy pocos registros**, o al menos marca explícitamente que son pocos.

Puedes filtrar los grupos pequeños directamente con `.groupby().filter()`:

In [ ]:
socios.groupby("ciudad").filter(lambda grupo: len(grupo) >= 5)["ciudad"].value_counts()

Lo mismo pasa con los periodos. Si cuentas las inscripciones por mes, muchos meses tienen muy pocos socios, así que una caída de 6 a 3 inscripciones puede ser ruido. Antes de decir que algo "subió" o "bajó", pregúntate cuántos registros hay detrás de cada número.

Recuerda también la advertencia de la Sección 3: el valor 53 de la mediana aparece por la imputación de la Clase 8, no porque los datos reales se concentren ahí. Un resultado repetido en todos los grupos merece sospecha antes de convertirse en conclusión.

## 10. Reto: cuadro de indicadores segmentado

Ahora te toca a ti. Construye **un cuadro de indicadores** que responda una pregunta de negocio, sobre `membresias_gimnasio_limpio.csv` o sobre el dataset propio que limpiaste en la Clase 8.

Tu entregable tiene tres partes:

1. **Un cuadro (`groupby`/`agg`)** con al menos **3 indicadores** (por ejemplo: conteo, promedio o mediana de una columna numérica, y una participación con `transform`) para **al menos 2 segmentos** (por ejemplo: ciudad, plan, grupo de edad, activo).
2. **Orden y tamaño de muestra**: ordena el cuadro según la pregunta y deja visible el conteo de cada grupo. Filtra o marca los grupos con menos de 5 registros.
3. **Tres hallazgos** escritos en la celda de markdown de abajo, cada uno con su número y el tamaño de muestra que lo respalda.

In [ ]:
# TODO: construye tu cuadro de indicadores (groupby + agg, con al menos 3 indicadores)


In [ ]:
# TODO: ordena el cuadro y marca o filtra los grupos con menos de 5 registros


**Mi pregunta de negocio:** *(escríbela aquí)*

**Hallazgo 1:** *(número, grupo, tamaño de muestra y qué significa)*

**Hallazgo 2:** *(número, grupo, tamaño de muestra y qué significa)*

**Hallazgo 3:** *(número, grupo, tamaño de muestra y qué significa)*

**Qué no me atrevo a concluir todavía, y por qué:** *(un grupo con pocos datos, un valor imputado, una comparación sin control)*

## Cierre

Hoy pasaste de preguntas sobre la tabla completa a preguntas sobre **segmentos**: `groupby` te permite dividir, aplicar y combinar en una sola instrucción. Las herramientas clave fueron `.agg()` con nombre, `.unstack()` y `pd.crosstab()` para cruzar categorías, `.transform()` para participaciones, y el hábito de siempre mirar el tamaño de cada grupo antes de leer un resultado.

Guarda una copia de este notebook en tu Google Drive antes de cerrar Colab.

En la **Clase 10: Merge, concatenación e integración de fuentes** vas a unir tablas distintas por una llave. Un caso natural es unir `socios` con una tabla de planes que tenga el precio y la duración de cada plan, en vez de escribir el diccionario a mano como hicimos en la Clase 8.